# TECHPULSE-AI: DistilBERT Severity Training on Google Colab

This notebook fine-tunes the TECHPULSE-AI severity classifier on a Colab GPU and publishes the result to Hugging Face Hub.

Before running it, add an `HF_TOKEN` secret with **write** access in the Colab Secrets panel and upload `techpulse_dataset.parquet` to Google Drive.

In [ ]:
# In Colab: Runtime > Change runtime type > T4 GPU
!nvidia-smi

In [ ]:
!git clone https://github.com/amosagekouassi-source/TECHPULSE-AI.git
%cd /content/TECHPULSE-AI
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive')
login(token=userdata.get('HF_TOKEN'))

In [ ]:
from pathlib import Path

DATASET_PATH = Path('/content/drive/MyDrive/TECHPULSE-AI/data/processed/techpulse_dataset.parquet')
HF_MODEL_ID = 'YOUR_HUGGING_FACE_USERNAME/techpulse-distilbert-severity'

assert DATASET_PATH.is_file(), f'Dataset not found: {DATASET_PATH}'
assert not HF_MODEL_ID.startswith('YOUR_'), 'Set your Hugging Face username first.'

In [ ]:
from app.classifier.config import ClassifierConfig
from app.classifier.train import train

config = ClassifierConfig(
    dataset_path=DATASET_PATH,
    model_output_dir=Path('/content/TECHPULSE-AI/models/distilbert_severity'),
    device='cuda',
    huggingface_model_id=HF_MODEL_ID,
    huggingface_private=True,
)
summary = train(config)
summary

In [ ]:
from app.classifier.predict import SeverityPredictor

predictor = SeverityPredictor(HF_MODEL_ID)
predictor.predict('Remote code execution vulnerability in Apache')